# JAX Live output: membership, ordering and exact controls

## Goal

Compare the output boundary with the original NumPy selection operations and
PyCBC ranking. The deterministic examples isolate background order, tied
trigger membership and ranking before checking all supplied trigger columns
and HDF datasets. The original controls restore exact shape, dtype and bytes.
Upstream filtering, template generation, conditioning and veto inputs are
held fixed; this notebook does not run a complete search or MPI.

## Setup

Use a PyCBC installation with JAX, NumPy, h5py and pandas. The notebook selects
the first available NVIDIA GPU, otherwise a CPU. Change `device_choice` to
validate another supported device. The displayed library versions and hardware
describe this execution. All inputs are generated here; output files exist
only in a temporary directory.

In [1]:
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory
import warnings

warnings.filterwarnings("ignore", message="Wswiglal-redir-stdio:")

import h5py
import jax
import numpy as np
import pandas as pd
from IPython.display import display

from pycbc import scheme
from pycbc.events import ranking
from pycbc.events.live_output_jax import (
    JAXLiveOutputWriter, collect_live_background_jax,
    snapshot_live_output,
)
from pycbc.types import FrequencySeries

device_choice = "cuda:0" if any(d.platform == "gpu" for d in jax.devices()) else "cpu"
with scheme.JAXScheme(device_choice) as context:
    hardware = context.jax_device.device_kind
    assert jax.config.x64_enabled
print({"NumPy": np.__version__, "JAX": jax.__version__,
       "JAXlib": version("jaxlib"), "h5py": h5py.__version__,
       "device": device_choice, "hardware": hardware})

def host(value):
    return np.asarray(jax.device_get(value))

def exact(actual, expected):
    actual, expected = host(actual), np.asarray(expected)
    assert actual.shape == expected.shape
    assert actual.dtype == expected.dtype
    assert actual.tobytes() == expected.tobytes()

def comparison(name, actual, expected):
    actual, expected = host(actual), np.asarray(expected)
    assert actual.shape == expected.shape and actual.dtype == expected.dtype
    return {"calculation": name, "dtype": str(actual.dtype),
            "different elements": int(np.count_nonzero(actual != expected)),
            "maximum absolute difference": float(np.max(np.abs(actual - expected))),
            "exact bytes": actual.tobytes() == expected.tobytes()}

{'NumPy': '2.5.2', 'JAX': '0.11.1', 'JAXlib': '0.11.1', 'h5py': '3.16.0', 'device': 'cpu', 'hardware': 'cpu'}


## Steps

### 1. Background membership and stored order

Both paths select the same seven largest values in this example. A partition
does not require their internal order to match. NumPy uses introselect, while
JAX's partition uses top-k operations; the recorded arrays expose that order
difference without changing the input values. `live_output_selection` invokes
the original NumPy expression and restores its stored order.

Sources: [NumPy partition](https://numpy.org/doc/stable/reference/generated/numpy.partition.html),
[JAX partition](https://docs.jax.dev/en/latest/_autosummary/jax.numpy.partition.html).

In [2]:
background = np.random.default_rng(812).normal(size=64).astype(np.float32)
background_count = 7
original_background = -np.partition(-background, background_count)[:background_count]
with scheme.JAXScheme(device_choice):
    default_background = collect_live_background_jax(background, background_count)
with scheme.JAXScheme(device_choice, reference_operations=("live_output_selection",)):
    reference_background = collect_live_background_jax(background, background_count)
exact(np.sort(default_background), np.sort(original_background))
exact(reference_background, original_background)
display(pd.DataFrame({"original stored order": original_background,
                      "default JAX stored order": default_background,
                      "original-control order": reference_background}))
print({"same membership": True,
       "default order exact": default_background.tobytes() == original_background.tobytes(),
       "original-control order exact": True})

,original stored order,default JAX stored order,original-control order
0,2.400772,2.422873,2.400772
1,2.412510,2.412510,2.412510
2,2.094041,2.400772,2.094041
3,2.422873,2.166298,2.422873
4,2.166298,2.094041,2.166298
5,2.086637,2.086637,2.086637
6,1.925001,1.925001,1.925001


{'same membership': True, 'default order exact': False, 'original-control order exact': True}


### 2. Equal SNR values can change selected rows

With reduced chi-square equal to one, NewSNR equals SNR exactly. This removes
ranking roundoff from the fixture. Original Live output uses NumPy's default
quicksort; JAX defaults to a stable ascending sort. Both reverse that order
before taking the largest entries, then form a sorted index union. Equal
scores therefore need not select the same rows. The public snapshot interface
exposes the actual output selection, rather than a separate JAX imitation.

Sources: [NumPy argsort](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html),
[JAX argsort](https://docs.jax.dev/en/latest/_autosummary/jax.numpy.argsort.html).

In [3]:
tied_snr = np.tile(np.array([4., 9., 6.], dtype=np.float32), 12)
tied_chisq = np.ones_like(tied_snr)
loudest_count = 5
with scheme.CPUScheme():
    original_tied_ranking = ranking.newsnr(tied_snr, tied_chisq)
original_loudest = np.union1d(np.argsort(original_tied_ranking)[::-1][:loudest_count],
                             np.argsort(tied_snr)[::-1][:loudest_count])
selection_columns = {"H1": {"snr": tied_snr, "chisq": tied_chisq}}
with scheme.JAXScheme(device_choice):
    exact(ranking.newsnr(tied_snr, tied_chisq), original_tied_ranking)
    default_snapshot = snapshot_live_output(
        "example.hdf", {}, selection_columns, store_loudest_index=loudest_count)
with scheme.JAXScheme(device_choice, reference_operations=("live_output_selection",)):
    reference_snapshot = snapshot_live_output(
        "example.hdf", {}, selection_columns, store_loudest_index=loudest_count)
default_loudest = host(default_snapshot.loudest["H1"])
exact(reference_snapshot.loudest["H1"], original_loudest)
exact(tied_snr[default_loudest], tied_snr[original_loudest])
display(pd.DataFrame({"original row": original_loudest,
                      "default JAX row": default_loudest,
                      "original-control row": reference_snapshot.loudest["H1"]}))
print({"ranking exact": True,
       "default row selection exact": default_loudest.tobytes() == original_loudest.tobytes(),
       "original-control selection exact": True})

,original row,default JAX row,original-control row
0,1,22,1
1,16,25,16
2,19,28,19
3,22,31,22
4,31,34,31


{'ranking exact': True, 'default row selection exact': False, 'original-control selection exact': True}


### 3. Ranking is an independent calculation

Both ranking implementations promote the inputs to float64. Here chi-square
values span the reweighting boundary and exercise powers and multiplication.
The table reports actual equality or roundoff; a default mismatch is not
required. `newsnr` restores the original formula without changing output
selection. Conversely, `live_output_selection` restores sorting while leaving
the default JAX ranking active. Combining them restores both boundaries.

In [4]:
ranking_snr = np.array([4.5, 7.2, 8.1, 12.4, 9.9, 6.3, 10.8], dtype=np.float32)
ranking_chisq = np.array([0.5, 1., 1.0001, 1.7, 3.2, 8., 12.], dtype=np.float32)
with scheme.CPUScheme():
    original_ranking = ranking.newsnr(ranking_snr, ranking_chisq)
ranking_comparisons = []
for label, operations in (
        ("default JAX", ()),
        ("output selection only", ("live_output_selection",)),
        ("original ranking only", ("newsnr",)),
        ("both original controls", ("newsnr", "live_output_selection"))):
    with scheme.JAXScheme(device_choice, reference_operations=operations):
        values = ranking.newsnr(ranking_snr, ranking_chisq)
    ranking_comparisons.append(comparison(label, values, original_ranking))
    if "newsnr" in operations:
        exact(values, original_ranking)
display(pd.DataFrame(ranking_comparisons).set_index("calculation"))

,dtype,different elements,maximum absolute difference,exact bytes
calculation,,,,
default JAX,float64,0,0.0,True
output selection only,float64,0,0.0,True
original ranking only,float64,0,0.0,True
both original controls,float64,0,0.0,True


## Checks

### 4. Complete supplied columns and HDF output

Hold the upstream trigger fields fixed and compose both original controls.
The writer receives numerical device columns, host detector labels, the
selected background, a PSD and gate metadata. A deferred snapshot retains
its device arrays until the ordered writer collects them. `drain()` and
`close()` finish the files before they are read.

Every supplied numerical trigger column, the selected indices, background,
PSD and gate fields must match their expected dtype, shape and bytes. Unicode
labels use the original list-of-encoded-bytes conversion: their HDF string
metadata and element bytes are checked, rather than object-array pointers.
PSD spacing, epoch, compression and file attributes are checked separately. This proves
the output boundary for these inputs, without assuming upstream search
calculations already match.

In [5]:
size = len(tied_snr)
columns = {
    "snr": tied_snr.copy(), "chisq": tied_chisq.copy(),
    "chisq_dof": np.full(size, 16, dtype=np.int32),
    "sg_chisq": np.full(size, 1.25, dtype=np.float32),
    "sigmasq": np.linspace(1., 2., size, dtype=np.float32),
    "coa_phase": np.linspace(-1., 1., size, dtype=np.float32),
    "end_time": 1.e9 + np.arange(size, dtype=np.float64) * 0.125,
    "template_id": np.arange(size, dtype=np.int64),
}
labels = np.array(["H1", "L1"])
psd_values = np.linspace(1., 3., 17, dtype=np.float32)
gate_dtype = [("center_time", float), ("zero_half_width", float), ("taper_width", float)]
gates = [(1.e9 + 0.5, 0.1, 0.2)]
attributes = {"num_live_detectors": 1, "description": "Live output example"}
output_checks = []
with TemporaryDirectory() as directory:
    filename = Path(directory) / "output.hdf"
    with scheme.JAXScheme(device_choice, reference_operations=("newsnr", "live_output_selection")):
        device_columns = {key: jax.device_put(value, scheme.mgr.state.jax_device)
                          for key, value in columns.items()}
        selected_background = collect_live_background_jax(background, background_count)
        snapshot = snapshot_live_output(
            filename, attributes, {"H1": device_columns},
            raw_results={"background/stat": selected_background,
                         "foreground/ifos": labels, "foreground/network": "H1-L1"},
            store_loudest_index=loudest_count,
            store_psd={"H1": FrequencySeries(psd_values, delta_f=0.5, epoch=1.e9)},
            gates={"H1": gates}, defer_readback=True)
        assert snapshot.device_pending
        for key, expected in columns.items():
            exact(snapshot.results["H1"][key], expected)
        writer = JAXLiveOutputWriter(queue_depth=1)
        try:
            writer.submit(snapshot)
            writer.drain()
        finally:
            writer.close()
    expected_datasets = {f"H1/{key}": value for key, value in columns.items()}
    expected_datasets.update({
        "H1/loudest": original_loudest,
        "background/stat": original_background,
        "H1/psd": psd_values,
        "H1/gates": np.array(gates, dtype=gate_dtype),
    })
    with h5py.File(filename, "r") as stored:
        for path, expected in expected_datasets.items():
            actual = stored[path][:]
            exact(actual, expected)
            dtype_label = "float64 gate fields" if actual.dtype.fields else str(actual.dtype)
            output_checks.append({"dataset": path, "shape": str(actual.shape),
                                  "dtype": dtype_label, "exact bytes": True})
        assert stored["foreground/network"].asstr()[()] == "H1-L1"
        label_dataset = stored["foreground/ifos"]
        label_type = h5py.check_string_dtype(label_dataset.dtype)
        assert label_dataset.shape == labels.shape and label_dataset.dtype == np.dtype(object)
        assert label_type.encoding == "ascii" and label_type.length is None
        assert label_dataset[:].tolist() == [item.encode() for item in labels]
        assert dict(stored.attrs) == attributes
        psd_dataset = stored["H1/psd"]
        assert psd_dataset.attrs["delta_f"] == 0.5 and psd_dataset.attrs["epoch"] == 1.e9
        assert psd_dataset.compression == "gzip" and psd_dataset.compression_opts == 9
        assert psd_dataset.shuffle
display(pd.DataFrame(output_checks).set_index("dataset"))
print("All supplied columns, output selection, background and numeric HDF datasets are byte-exact.")
print("Text, file attributes and PSD metadata/compression also match their expected contracts.")

,shape,dtype,exact bytes
dataset,,,
H1/snr,"(36,)",float32,True
H1/chisq,"(36,)",float32,True
H1/chisq_dof,"(36,)",int32,True
H1/sg_chisq,"(36,)",float32,True
H1/sigmasq,"(36,)",float32,True
H1/coa_phase,"(36,)",float32,True
H1/end_time,"(36,)",float64,True
H1/template_id,"(36,)",int64,True
H1/loudest,"(5,)",int64,True


All supplied columns, output selection, background and numeric HDF datasets are byte-exact.
Text, file attributes and PSD metadata/compression also match their expected contracts.


## Next steps

The default examples isolate order and tie membership independently of
floating-point ranking. `live_output_selection` restores original output
selection; `newsnr` restores ranking independently. Their composition preserves
the full supplied output, with host transfer only at the selected reference
and serialization boundaries.

For upstream trigger and veto calculations, use the
[search numerical comparison notebook](jax_search_numerical_differences.ipynb)
and the [JAX Live guide](../../docs/jax_live.rst). An exact output example does
not establish equivalence for other data, devices, MPI scheduling or an entire
search. Rerun the assertions with the intended library versions and hardware.